# Exercise 15: Text-to-Speech Conversation

**Goals**

In this notebook, you will:
- Send a short text query to GPT.
- Receive a concise AI reply.
- Convert both the user input and the AI reply into speech (TTS).
- Play back the generated audio responses in the notebook.

This is a text-to-speech exercise: the input is typed text, not microphone audio. Install or update the Python package with `%pip install -U openai` if needed, and set `OPENAI_API_KEY` in your environment. Do not paste your API key into this notebook.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
from pathlib import Path
from tempfile import TemporaryDirectory
from IPython.display import Audio, display

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
client = OpenAI()
CHAT_MODEL = "gpt-6-luna"
TTS_MODEL = "gpt-4o-mini-tts"

In [ ]:
# ----------------- Step 1: LLM Response -----------------
def chat(text_input: str) -> str:
    res = client.responses.create(
        model=CHAT_MODEL,
        instructions="Reply in at most two concise sentences.",
        input=text_input,
        reasoning={"effort": "none"},
        max_output_tokens=150,
    )
    reply = res.output_text.strip()
    if not reply:
        raise RuntimeError("The model returned an empty reply.")
    return reply

In [ ]:
# ----------------- Step 2: Text-to-Speech -----------------
def tts_and_display(text: str):
    if not text or not text.strip():
        return

    with TemporaryDirectory() as temp_dir:
        out_path = Path(temp_dir) / "speech.wav"
        with client.audio.speech.with_streaming_response.create(
            model=TTS_MODEL,
            voice="alloy",
            input=text,
            response_format="wav",
        ) as resp:
            resp.stream_to_file(out_path)
        display(Audio(filename=str(out_path)))

In [ ]:
def main():
    user_text = "How do you like living in Hawaii? Have you explored mountains?"

    print(f"\n[You] {user_text}")
    tts_and_display(user_text)

    reply = chat(user_text)
    print(f"[LLM] {reply}")

    tts_and_display(reply)

if __name__ == "__main__":
    main()